In [41]:
import operator
from typing import Annotated, List, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, BaseMessage
from langchain_core.tools import tool

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Send, interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph.message import add_messages

In [42]:
class PlotState(BaseModel):
    target: str = Field(description="The target for whom the plot is being hatched.")
    culprit: str = Field(description="The culprit behind the plot.")
    severity: int = Field(description="The severity of the plot on a scale of 1 to 10.")

# State for the parallel sub-graph instances
class SubState(TypedDict):
    region: str
    rumor: str
    verified_plots: List[dict] 

# State for the primary parent graph
class RumorState(TypedDict):
    raw_rumors: List[dict]
    verified_plots: Annotated[List[dict], operator.add]
    evaluate_treason: str
    evaluated_culprits: List[str]
    pending_royal_decree: str
    kings_decision: str
    # CRITICAL: Tools require the add_messages reducer to track tool calls & returns
    messages: Annotated[List[BaseMessage], add_messages] 

llm = ChatOllama(model="mistral")

In [43]:
@tool("execute_tool")
def execute_tool(culprits: List[str]) -> str:
    """This tool is used to execute culprits. It takes a list of culprit names as input."""
    return f"THE ROYAL EXECUTIONER REPORT: The following culprits have been publicly executed: {', '.join(culprits)}."

@tool("assassinate_tool")
def assassinate_tool(culprits: List[str]) -> str:
    """This tool is used to quietly assassinate culprits. It takes a list of culprit names as input."""
    return f"THE MASTER OF WHISPERS REPORT: The target targets have been taken care of. The following culprits are dead: {', '.join(culprits)}."

tools = [execute_tool, assassinate_tool]
tool_node = ToolNode(tools)

In [44]:
llm_with_tools = llm.bind_tools(tools)

In [45]:
def verify_rumor_node(state: SubState) -> dict:
    prompt = f"""
    you are a trusted advisor to the hand of the king. Your job is to verify the rumors that are being circulated in the region : {state['region']}. The rumor that you need to verify is : {state['rumor']}. You need to verify the target of the plot, the culprit behind the plot and the severity of the plot on a scale of 1 to 10.
    Be extremely concise.
    """
    response = llm.with_structured_output(PlotState).invoke([SystemMessage(content=prompt)])
    return {'verified_plots': [response.model_dump()]}

In [46]:
sub = StateGraph(SubState)
sub.add_node("verify_rumor", verify_rumor_node)
sub.add_edge(START, "verify_rumor")
sub.add_edge("verify_rumor", END)

intel_subgraph = sub.compile()

In [47]:
def gather_intel_node(state: RumorState):
    rumors = state.get("raw_rumors", [])
    return [
        Send("intel_subgraph_node", SubState(region=r['region'], rumor=r['rumor'], verified_plots=[])) 
        for r in rumors
    ]

class TreasonEvaluation(BaseModel):
    statement: str = Field(description="A concise statement evaluating the treasonous nature of the plots.")
    culprits: List[str] = Field(description="The names of the culprits behind the plots.")

def evaluate_treason_node(state: RumorState) -> dict:
    verified_plots = state.get('verified_plots', [])
    pending_royal_decree = "No action needed"
    combined_report: List[str] = []
    
    for plot in verified_plots:
        combined_report.append(f"Plot against {plot['target']} by {plot['culprit']} with severity {plot['severity']}")
        if int(plot['severity']) >= 7:
            pending_royal_decree = "Royal decree pending"
            
    information = "\n".join(combined_report)
    prompt = f"""You are the hand of the king. Evaluate the treasonous nature of these plots, provide a concise statement, and isolate the names of the culprits. Information:\n{information}"""
    
    response = llm.with_structured_output(TreasonEvaluation).invoke([SystemMessage(content=prompt)])
    return {
        'evaluate_treason': response.statement, 
        'evaluated_culprits': response.culprits, 
        'pending_royal_decree': pending_royal_decree
    }

def await_royal_command_node(state: RumorState) -> dict:
    if state.get('pending_royal_decree') == "Royal decree pending":
        decision = interrupt({
            'type': 'input',
            'message': f"The culprits behind the severe plots are {state['evaluated_culprits']}. What is your royal decree?",
            "instructions": "Provide input as: {'input': 'pardon' | 'assassinate' | 'execute'}"
        })
        return {'kings_decision': decision['input']}
    else:
        return {'kings_decision': "No action needed"}

def king_router(state: RumorState) -> Literal["pardon", "commander"]:
    if state.get('kings_decision') not in ["assassinate", "execute"]:
        return "pardon"
    else:
        return "commander"

def commander_node(state: RumorState) -> dict:
    # Build a strict instruction prompt forcing the LLM to call the tool corresponding to the decree
    prompt = f"""You are the commander of the royal army. 
    The king has issued an official decree: "{state['kings_decision']}" targeting these exact culprits: {state['evaluated_culprits']}.
    
    You must call the appropriate tool immediately to execute this command. 
    - If the decision is 'execute', use 'execute_tool'.
    - If the decision is 'assassinate', use 'assassinate_tool'.
    Pass the exact list of culprits: {state['evaluated_culprits']} to the chosen tool.
    """
    response = llm_with_tools.invoke([SystemMessage(content=prompt)])
    return {"messages": [response]}

In [48]:
checkpointer = InMemorySaver()
graph = StateGraph(RumorState)

# Node registration
graph.add_node("intel_subgraph_node", intel_subgraph)
graph.add_node("evaluate_treason", evaluate_treason_node)
graph.add_node("await_royal_command", await_royal_command_node)
graph.add_node("commander", commander_node)
graph.add_node("tools", tool_node)

# Map-Reduce Sequence
graph.add_conditional_edges(START, gather_intel_node, ["intel_subgraph_node"])
graph.add_edge("intel_subgraph_node", "evaluate_treason")
graph.add_edge("evaluate_treason", "await_royal_command")

# Human branch check
graph.add_conditional_edges("await_royal_command", king_router, {
    "pardon": END,
    "commander": "commander"
})

# Autonomous Tool Execution Routing
graph.add_conditional_edges("commander", tools_condition)
graph.add_edge("tools", END) # The tool execution result outputs cleanly to messages history, then finishes.

final_graph = graph.compile(checkpointer=checkpointer)

In [50]:
CONFIG = {'configurable': {'thread_id': 'royal-court-thread-v3'}}

# Test case with highly severe rumors to guarantee severity triggers the interrupt
initial_state = {
    "raw_rumors": [
        {"region": "Essos", "rumor": "A rogue cell is smuggling wildfire into the castle vaults to blow up the tower."},
        {"region": "Westeros", "rumor": "An assassin plot is brewing against the King during the high tourney feast."}
    ],
    "messages": []
}

print("Initiating Spies and Whispers flow...")
output = final_graph.invoke(initial_state, CONFIG)

# Check for Human-In-The-Loop Interrupt status
get_snapshot = final_graph.get_state(CONFIG)

if get_snapshot.tasks and get_snapshot.tasks[0].interrupts:
    interrupt_payload = get_snapshot.tasks[0].interrupts[0].value
    print("\n" + "⚔️ " * 15)
    print("👑 COURT INTERRUPT: THE KING'S AUDIENCE REQUIRED 👑")
    print(interrupt_payload['message'])
    print("⚔️ " * 15 + "\n")

    # Change this choice between 'execute' or 'assassinate' to watch the LLM switch tools dynamically!
    user_choice = "assassinate"  
    print(f"Communicating Royal Decree to the Hand: {user_choice}\n")
    
    king_decision = {"input": user_choice.lower()}
    
    # Resume the graph with the command payload
    final_output = final_graph.invoke(Command(resume=king_decision), config=CONFIG)
    
    print("="*50)
    print("📜 FINAL CHRONICLE RESULTS 📜")
    print(f"King's Final Choice: {final_output.get('kings_decision')}")
    print("\nLast Agent Message Logs:")
    # The last message in the list will be the ToolMessage returned by our tool execution node!
    print(final_output.get('messages')[-1].content)
    print("="*50)
else:
    print("Workflow processed without needing high-level royal intervention.")

Initiating Spies and Whispers flow...

⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ 
👑 COURT INTERRUPT: THE KING'S AUDIENCE REQUIRED 👑
The culprits behind the severe plots are []. What is your royal decree?
⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ ⚔️ 

Communicating Royal Decree to the Hand: assassinate

📜 FINAL CHRONICLE RESULTS 📜
King's Final Choice: assassinate

Last Agent Message Logs:
 Q: What is 793.2 rounded to the nearest one hundred?
A: 800
